# Non-Linear Regression — Practice Set 1 🎯
## Earnings & Age Dataset

### Rules (same as real quiz)
- `alpha = 0.05` unless stated otherwise
- `random_state = 617`
- Use `train` sample unless stated otherwise
- Do NOT round answers
- No commas (`100000` not `100,000`), no units, lead with 0 (`0.34` not `.34`)

### About the Data
This dataset (`earning.csv`) describes earnings and personal characteristics of a sample of workers.
- `earn`: Annual earnings in dollars
- `age`: Age in years
- `height`: Height in inches
- `weight`: Weight in pounds
- `gender`: Male or Female
- `education`: Years of education

> **Note**: Use `earning.csv` with this assignment. The relationship between age and earnings is expected to be non-linear (earnings rise then fall with age).

In [ ]:
# Setup — run this first
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.formula.api import ols

data = pd.read_csv('earning.csv')
train = data.sample(frac=0.7, random_state=617)
test = data.drop(labels=train.index)
print('Train:', train.shape, '| Test:', test.shape)

---
## ❓ Question 1
> Read in `earning.csv` and split into train (70%) and test (30%) using `data.sample()` with `random_state=617`.
>
> Construct a scatter plot with `age` on the x-axis and `earn` on the y-axis.
> **Does the relationship between age and earnings appear to be linear or non-linear? What shape does it take?**

**Expected answer:** Non-linear / inverted-U shape (earnings rise then fall with age)

In [ ]:
# Your code here
sns.scatterplot(data=train, x='age', y='earn', s=10, alpha=0.6)
plt.show()

---
## ❓ Question 2
> Fit a **linear regression** to predict `earn` from `age`. Call this `model_linear`.
>
> Then fit a **degree-2 polynomial regression** to predict `earn` from `age`. Call this `model_poly2`.
>
> **What is the R² of `model_poly2`?**

💡 Hint: Use `I(age**2)` in the statsmodels formula syntax.

In [ ]:
# Linear
model_linear = ols('earn ~ age', data=train).fit()
print('Linear R²:', model_linear.rsquared)

# Degree-2 polynomial (must include age AND age²)
model_poly2 = ols('earn ~ age + I(age**2)', data=train).fit()
print('Poly-2 R²:', model_poly2.rsquared)

---
## ❓ Question 3
> Based on `model_poly2`, **is the quadratic term (`I(age**2)`) statistically significant at α = 0.05?**
>
> **What does the sign of the quadratic coefficient tell you about the shape of the relationship?**

In [ ]:
print(model_poly2.summary())
print('\np-value of age²:', model_poly2.pvalues['I(age ** 2)'])
print('Coefficient of age²:', model_poly2.params['I(age ** 2)'])
# Negative coefficient → inverted-U (earnings peak then decline)

---
## ❓ Question 4
> Fit a **degree-3 polynomial regression**. Call this `model_poly3`. (Remember: must include `age`, `age²`, AND `age³`.)
>
> **What is the train RMSE for `model_poly3`?**

In [ ]:
model_poly3 = ols('earn ~ age + I(age**2) + I(age**3)', data=train).fit()

pred_train = model_poly3.predict()
rmse_train = np.sqrt(np.mean((train.earn - pred_train)**2))
print('Poly-3 Train RMSE:', rmse_train)

---
## ❓ Question 5
> Compare the **test RMSE** for `model_linear`, `model_poly2`, and `model_poly3`.
>
> **Which model has the best (lowest) test RMSE?**

In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred)**2))

results = pd.DataFrame({
    'model': ['linear', 'poly2', 'poly3'],
    'train_rmse': [
        rmse(train.earn, model_linear.predict()),
        rmse(train.earn, model_poly2.predict()),
        rmse(train.earn, model_poly3.predict()),
    ],
    'test_rmse': [
        rmse(test.earn, model_linear.predict(test)),
        rmse(test.earn, model_poly2.predict(test)),
        rmse(test.earn, model_poly3.predict(test)),
    ]
}).set_index('model')
print(results)
print('\nBest test RMSE:', results.test_rmse.idxmin())

---
## ❓ Question 6
> Create a **step function** model by binning `age` into 4 groups: `[0,30)`, `[30,45)`, `[45,60)`, `[60,100)`. Label them `young`, `early_mid`, `late_mid`, `senior`.
>
> Fit a regression predicting `earn` from `age_bin`. Call this `model_step`.
>
> **What is the R² of `model_step`?**

In [ ]:
train = train.copy()
test = test.copy()

train['age_bin'] = pd.cut(train['age'],
                          bins=[0, 30, 45, 60, 100],
                          labels=['young','early_mid','late_mid','senior'])
test['age_bin'] = pd.cut(test['age'],
                         bins=[0, 30, 45, 60, 100],
                         labels=['young','early_mid','late_mid','senior'])

model_step = ols('earn ~ age_bin', data=train).fit()
print('Step R²:', model_step.rsquared)
print(model_step.params)

---
## ❓ Question 7
> Based on `model_step`, **on average how much more do `late_mid` workers earn compared to `young` workers?**
>
> 💡 Hint: Look at the coefficient for `age_bin[T.late_mid]`.

In [ ]:
model_step.params['age_bin[T.late_mid]']
# This is the difference vs the reference group (young)

---
## ❓ Question 8
> Use sklearn's `PolynomialFeatures` to create a degree-2 feature matrix from `age` only.
>
> Fit a `LinearRegression` model. Call this `model_poly2_sk`.
>
> **What is the predicted earn for the first observation in the train sample?**

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

# Create polynomial features from age
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_p = poly.fit_transform(train[['age']])
X_test_p  = poly.transform(test[['age']])

model_poly2_sk = LinearRegression()
model_poly2_sk.fit(X_train_p, train.earn)

pred_train = model_poly2_sk.predict(X_train_p)
print('First observation predicted earn:', pred_train[0])

---
## ❓ Question 9
> Using `model_poly2_sk`, **what is the test RMSE?**

In [ ]:
pred_test = model_poly2_sk.predict(X_test_p)
rmse_test = root_mean_squared_error(test.earn, pred_test)
print('Test RMSE:', rmse_test)

---
## ❓ Question 10
> Now add `gender` as an additional feature alongside the degree-2 polynomial of `age`.
> Dummy-encode `gender` with `drop_first=True` and align test columns to train.
>
> Fit this expanded model. Call it `model_poly2_full`.
>
> **Is the test RMSE of `model_poly2_full` lower than `model_poly2_sk`? By how much?**

In [ ]:
# Create polynomial features for age
X_train_age = poly.transform(train[['age']])
X_test_age  = poly.transform(test[['age']])

# Dummy encode gender
gender_train = pd.get_dummies(train[['gender']], drop_first=True).astype(float)
gender_test  = pd.get_dummies(test[['gender']],  drop_first=True).astype(float)
gender_test  = gender_test.reindex(columns=gender_train.columns, fill_value=0)

# Combine
X_tr = np.hstack([X_train_age, gender_train.values])
X_te = np.hstack([X_test_age,  gender_test.values])

model_poly2_full = LinearRegression().fit(X_tr, train.earn)

rmse_full_train = root_mean_squared_error(train.earn, model_poly2_full.predict(X_tr))
rmse_full_test  = root_mean_squared_error(test.earn,  model_poly2_full.predict(X_te))
print('Full model Train RMSE:', rmse_full_train)
print('Full model Test RMSE: ', rmse_full_test)
print('Improvement over poly2_sk:', rmse_test - rmse_full_test)

---
# ✅ Answer Key

| Q | Key Answer | What to Check |
|---|---|---|
| 1 | Non-linear, inverted-U | Shape of scatter plot |
| 2 | `model_poly2.rsquared` | Higher than linear R² |
| 3 | Significant (p<0.05); Negative → inverted-U | `pvalues['I(age ** 2)']` |
| 4 | `model_poly3` train RMSE | `np.sqrt(mean((earn-pred)²))` |
| 5 | Usually `poly2` or `poly3` | Lowest `test_rmse` |
| 6 | `model_step.rsquared` | R² of step function |
| 7 | `model_step.params['age_bin[T.late_mid]']` | Difference from reference (young) |
| 8 | `pred_train[0]` | First row prediction |
| 9 | `rmse_test` (sklearn poly2) | Test RMSE |
| 10 | Compare RMSEs | Adding gender should help |

## 🔑 Key Concepts Tested
- Polynomial terms must include ALL lower-degree terms
- `I(age**2)` syntax in statsmodels formula
- Negative quadratic coefficient → inverted-U relationship
- Step functions use `pd.cut()` + `ols()` like a categorical variable
- `PolynomialFeatures` in sklearn, followed by `LinearRegression`
- Model selection based on TEST RMSE, not train RMSE